In [1]:
def is_var(t):
    return isinstance(t, str) and t[:1].islower()

def substitute(t, theta):
    """Apply substitution theta to a term."""
    if is_var(t):
        return substitute(theta[t], theta) if t in theta else t
    if isinstance(t, tuple):
        return tuple(substitute(a, theta) for a in t)
    return t

def occurs(v, t, theta):
    t = substitute(t, theta)
    return v == t or (isinstance(t, tuple) and any(occurs(v, a, theta) for a in t))

def unify_var(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_var(t) and t in theta:
        return unify(v, theta[t], theta)
    if occurs(v, t, theta):
        return None
    return {**theta, v: t}

def unify(x, y, theta):
    """Return the most general unifier extending theta, or None."""
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None

facts = {
    ("American", "Robert"),
    ("Enemy", "A", "America"),
    ("Owns", "A", "T1"),
    ("Missile", "T1"),
}

rules = [
    ([("Missile", "x"), ("Owns", "A", "x")], ("Sells", "Robert", "x", "A")),
    ([("Missile", "x")], ("Weapon", "x")),
    ([("Enemy", "x", "America")], ("Hostile", "x")),
    ([("American", "p"), ("Weapon", "q"), ("Sells", "p", "q", "r"), ("Hostile", "r")],
     ("Criminal", "p")),
]

query = ("Criminal", "Robert")


def match(premises, known, theta):
    """Yield every substitution that makes all premises true in `known`."""
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for fact in known:
        t2 = unify(first, fact, theta)
        if t2 is not None:
            yield from match(rest, known, t2)

def forward_chain(facts, rules, query):
    known = set(facts)
    iteration = 0
    while True:
        iteration += 1
        new = {}
        for premises, conclusion in rules:
            for theta in match(premises, known, {}):
                fact = substitute(conclusion, theta)
                if fact not in known and fact not in new:
                    new[fact] = (premises, theta)
        if not new:
            return False
        print(f"\n--- Iteration {iteration} ---")
        for fact, (premises, theta) in new.items():
            used = " & ".join(str(substitute(p, theta)) for p in premises)
            print(f"{used} => {fact} {theta}")
        known |= set(new)
        if query in new or query in known:
            print(f"\nQuery {query} proved by forward chaining.")
            return True

if __name__ == "__main__":
    print("Initial facts:", *sorted(facts), sep="\n ")
    if not forward_chain(facts, rules, query):
        print("Query cannot be proved.")

Initial facts:
 ('American', 'Robert')
 ('Enemy', 'A', 'America')
 ('Missile', 'T1')
 ('Owns', 'A', 'T1')

--- Iteration 1 ---
('Missile', 'T1') & ('Owns', 'A', 'T1') => ('Sells', 'Robert', 'T1', 'A') {'x': 'T1'}
('Missile', 'T1') => ('Weapon', 'T1') {'x': 'T1'}
('Enemy', 'A', 'America') => ('Hostile', 'A') {'x': 'A'}

--- Iteration 2 ---
('American', 'Robert') & ('Weapon', 'T1') & ('Sells', 'Robert', 'T1', 'A') & ('Hostile', 'A') => ('Criminal', 'Robert') {'p': 'Robert', 'q': 'T1', 'r': 'A'}

Query ('Criminal', 'Robert') proved by forward chaining.
